# H&M Dataset — FOAF Decentralized MF Experiment

Mirrors the ML-100K experiment structure exactly:
- Same FOAF gradient-sharing method (user-stratified, random-sampling batch)
- Same graph topologies: Random-2-out, Random-5-out, Scale-Free, Cycle
- Same evaluation metric: RMSE on held-out test set
- Same baselines: CL (centralized), FL (FedAvg), GL (Gossip), LL (local)

**Data source**: Download `articles.csv`, `customers.csv`, `transactions_train.csv`  
from https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations

**Target**: purchase frequency `y_ij = count(customer_i, product_j)` — same as appendix.

**Pipeline order**:
1. Load & add week index
2. Filter by product type / customer activity / location
3. **Sample top-K users** (deterministic, ranked by total interaction count across all weeks)
4. **Temporal train/test split within the sampled subset** (week > 50 = train)
5. Deterministic per-user val split (last 20% of train, sorted by item id)
6. Re-index to contiguous 0-based integers

## 0. Imports

In [3]:
import pandas as pd
import numpy as np
import datetime
import math
import copy
import os
from dataclasses import dataclass, field
from typing import Dict, Optional
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import networkx as nx
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

## 1. Load & Filter

Loads raw H&M files, attaches `week` index, and applies the three domain filters
(top-6 product types, >=120-purchase customers, top-5000 postal codes).
No train/test split yet — `df` retains the full transaction history with `week` intact,
so user ranking in the next step is computed over all available history.

In [5]:
# ── Load raw files ──────────────────────────────────────────────────────────
DATA_DIR = "."   # change to path containing the three CSVs

df           = pd.read_csv(os.path.join(DATA_DIR, "transactions_train.csv"),
                           dtype={"article_id": str})
customer_df  = pd.read_csv(os.path.join(DATA_DIR, "customers.csv"),
                           dtype={"article_id": str, "product_code": str})
article_df   = pd.read_csv(os.path.join(DATA_DIR, "articles.csv"),
                           dtype={"article_id": str})

df["product_code"] = df["article_id"].map(
    article_df.set_index("article_id")["product_code"])

# ── Week index (0 = most recent week) ───────────────────────────────────────
df["t_dat"] = pd.to_datetime(df["t_dat"])
df["week"]  = (df["t_dat"].max() - df["t_dat"]).dt.days // 7

print("Raw transactions:", len(df))

Raw transactions: 31788324


In [6]:
# ── Filter: top-6 product types, >=120-purchase customers, top-5000 locations
# (exactly as in appendix Section 3.2 and H_M_Data.ipynb)
chosen_types = (
    article_df.groupby("product_type_name")["product_code"]
    .count().sort_values()[-6:].index
)
chosen_customers = (
    df.groupby("customer_id")["product_code"].nunique()
    .pipe(lambda s: s[s > 120]).index
)
location_customers = customer_df["postal_code"].value_counts()[1:5001].index
customer_df = customer_df[customer_df["postal_code"].isin(location_customers)]

chosen_articles = article_df[article_df["product_type_name"].isin(chosen_types)]
df = df[df["product_code"].isin(chosen_articles["product_code"])]
df = df[df["customer_id"].isin(chosen_customers)]
df = df[df["customer_id"].isin(customer_df["customer_id"])]

df.drop(["t_dat", "price", "article_id", "sales_channel_id"], axis=1, inplace=True)
df["bought"] = 1
df.reset_index(drop=True, inplace=True)

print(f"Filtered transactions : {len(df)}")
print(f"Unique customers      : {df['customer_id'].nunique()}")
print(f"Unique products       : {df['product_code'].nunique()}")
print(f"Week range            : {df['week'].min()} to {df['week'].max()}")

Filtered transactions : 216390
Unique customers      : 1765
Unique products       : 12782
Week range            : 0 to 104


## 2. Deterministic Sampling + Per-User Random Train/Test/Val Split

**Sampling**: aggregate all transactions to `(user, item)` purchase counts;
drop users below `MIN_TOTAL_INTERACTIONS`; rank by count descending
(ties broken by `customer_id` ascending); take top `TARGET_USERS`.
No randomness, no seed required for sampling.

**Train/test split** (within sampled data, cold-start-free):
For each user, sort their `(user, item)` rows by `product_code` ascending
and hold out the last `TEST_FRAC` fraction as test. Every user contributes
at least one row to train (floor of 1), so there are **no cold-start users**.
Items are induced from train rows only, so there are **no cold-start items**
in the test set either.

**Val split**: same per-user deterministic scheme applied to train rows;
last `VAL_FRAC` fraction sorted by `product_code` goes to val.

| TARGET_USERS | Scale | Use case |
|---|---|---|
| 300 | Fast debug | Topology sanity check |
| 500 | **ML-100K comparable** | Main experiment |
| 1000 | Medium | Larger-scale comparison |
| 1760 | Full | Full appendix replication |

In [8]:
# ── Configuration ────────────────────────────────────────────────────────────
TARGET_USERS           = 1000   # set to 1760 to use the full filtered dataset
MIN_TOTAL_INTERACTIONS = 20     # floor on total interactions per user
TEST_FRAC              = 0.25    # fraction of each user's interactions held out for test
VAL_FRAC               = 0.2    # fraction of each user's remaining train interactions for val
SPLIT_SEED             = 42     # seed for per-user random shuffle splits
SAMPLED_DATA_DIR       = "hm_sampled"


def build_hm_dataset(df_full,
                     target_users=1000,
                     min_total_interactions=20,
                     test_frac=0.2,
                     val_frac=0.1,
                     seed=42,
                     user_col="customer_id",
                     item_col="product_code",
                     target_col="bought"):
    """
    Deterministic sampling + per-user random train/test/val split.
    Guaranteed cold-start-free: every user and every item appears in train.

    Parameters
    ----------
    df_full : DataFrame
        Full filtered transaction table. Must contain user_col, item_col,
        target_col. The week column is NOT used — split is interaction-based.
    target_users : int
        Maximum number of users to keep.
    min_total_interactions : int
        Minimum unique (user, item) pairs required to include a user.
        Enforced before ranking so the floor is stable.
    test_frac : float
        Fraction of each user's interactions held out for test.
        Applied to sorted-by-item rows, so it is deterministic.
        Each user keeps at least 1 row in train (cold-start guard).
    val_frac : float
        Fraction of each user's remaining train interactions held out for val.
        Same random-shuffle scheme. Each user keeps at least 1 row in train
        after val is removed.
    seed : int
        Master seed for the per-user random shuffles. Fully reproducible
        for any fixed seed value.

    Returns
    -------
    train_df, val_df, test_df : DataFrames
        (user_idx, item_idx, count) with 0-based integer ids.
    meta : dict

    Cold-start guarantee
    --------------------
    - User cold start: impossible — every user contributes >= 1 row to train.
    - Item cold start: test and val contain only items present in train,
      because the item vocabulary is induced from train rows after splitting.
      Any test/val row whose item fell entirely into test/val is dropped.
    """

    # ── Step 1: aggregate to (user, item) → purchase count ───────────────────
    # Collapsing to unique (user, item) pairs is the natural unit for MF;
    # splitting at this level means we split observed preferences, not
    # individual purchase events.
    agg = (
        df_full.groupby([user_col, item_col])[target_col]
        .count().reset_index()
    )

    # ── Step 2: count unique items per user; apply floor ─────────────────────
    user_counts = (
        agg.groupby(user_col)[item_col]
        .count().rename("n_items").reset_index()
    )
    user_counts = user_counts[user_counts["n_items"] >= min_total_interactions]
    print(f"  Users passing floor (>= {min_total_interactions} unique items): "
          f"{len(user_counts)} / {agg[user_col].nunique()}")

    # ── Step 3: deterministic top-K (desc item count, asc user_id tie-break) ─
    user_counts = user_counts.sort_values(
        ["n_items", user_col], ascending=[False, True]
    )
    n_select = min(target_users, len(user_counts))
    if n_select < target_users:
        print(f"  WARNING: only {n_select} users available after floor "
              f"(requested {target_users})")
    sampled_users = user_counts.head(n_select)[user_col].values
    agg = agg[agg[user_col].isin(sampled_users)].copy()

    # ── Step 4: per-user random train/test split ─────────────────────────────
    # Shuffle each user's rows with a per-user draw from rng, then assign
    # the last round(test_frac * n) rows to test.
    # Cold-start guard: at most n-1 rows go to test so train always has >= 1.
    rng = np.random.default_rng(seed)

    train_rows, test_rows = [], []
    for _, grp in agg.groupby(user_col):
        shuffled = grp.sample(frac=1, random_state=int(rng.integers(1_000_000)))
        n_test   = min(max(1, round(len(shuffled) * test_frac)), len(shuffled) - 1)
        test_rows.append(shuffled.iloc[-n_test:])
        train_rows.append(shuffled.iloc[:-n_test])

    test_df    = pd.concat(test_rows).reset_index(drop=True)
    train_pool = pd.concat(train_rows).reset_index(drop=True)

    # ── Step 5: per-user random val split on train pool ───────────────────────
    # Same scheme applied independently to each user's train rows.
    # Cold-start guard: at most n-1 rows go to val so train always has >= 1.
    train_rows, val_rows = [], []
    for _, grp in train_pool.groupby(user_col):
        shuffled = grp.sample(frac=1, random_state=int(rng.integers(1_000_000)))
        n_val    = min(max(1, round(len(shuffled) * val_frac)), len(shuffled) - 1)
        val_rows.append(shuffled.iloc[-n_val:])
        train_rows.append(shuffled.iloc[:-n_val])

    val_df   = pd.concat(val_rows).reset_index(drop=True)
    train_df = pd.concat(train_rows).reset_index(drop=True)

    # ── Step 6: induce item vocabulary from train only ────────────────────────
    # Drop any val/test rows whose item does not appear in train.
    # This is the only source of item cold-start and is typically rare
    # because items are split per-user, not globally.
    train_items = set(train_df[item_col].unique())
    val_df  = val_df[val_df[item_col].isin(train_items)].copy()
    test_df = test_df[test_df[item_col].isin(train_items)].copy()

    # ── Step 7: re-index to contiguous 0-based integers ──────────────────────
    # User order : descending train interaction count (user-0 = most active).
    # Item order : ascending sorted product_code (fully deterministic).
    ordered_users = (
        train_df.groupby(user_col)[item_col].count()
        .sort_values(ascending=False).index
    )
    user2idx = {u: i for i, u in enumerate(ordered_users)}
    item2idx = {a: i for i, a in enumerate(sorted(train_items))}

    for df_ in [train_df, val_df, test_df]:
        df_[user_col] = df_[user_col].map(user2idx)
        df_[item_col] = df_[item_col].map(item2idx)
        df_.dropna(subset=[user_col, item_col], inplace=True)
        df_[[user_col, item_col]] = df_[[user_col, item_col]].astype(int)

    for df_ in [train_df, val_df, test_df]:
        df_.reset_index(drop=True, inplace=True)

    meta = {
        "n_users":           train_df[user_col].nunique(),
        "n_items":           train_df[item_col].nunique(),
        "n_train":           len(train_df),
        "n_val":             len(val_df),
        "n_test":            len(test_df),
        "n_users_with_test": test_df[user_col].nunique(),
    }
    return train_df, val_df, test_df, meta


# ── Save-or-load ──────────────────────────────────────────────────────────────
# Cache tag encodes every parameter that affects the output; no seed needed.
cache_tag  = f"u{TARGET_USERS}_m{MIN_TOTAL_INTERACTIONS}_t{int(TEST_FRAC*100)}_v{int(VAL_FRAC*100)}_s{SPLIT_SEED}"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

if all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]):
    # ── Fast path: reload from disk ───────────────────────────────────────────
    print(f"Loading cached dataset from {SAMPLED_DATA_DIR}/")
    train_df = pd.read_csv(train_path)
    val_df   = pd.read_csv(val_path)
    test_df  = pd.read_csv(test_path)
    meta_df  = pd.read_csv(meta_path)
    N_USERS  = int(meta_df.loc[meta_df["key"] == "n_users", "value"].iloc[0])
    N_ITEMS  = int(meta_df.loc[meta_df["key"] == "n_items", "value"].iloc[0])
    print(f"  Loaded : {N_USERS} users | {N_ITEMS} items")
    print(f"  Train  : {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

else:
    # ── Slow path: build from scratch, then save ──────────────────────────────
    print("Cache not found — running sampling + split pipeline...")

    train_df, val_df, test_df, meta = build_hm_dataset(
        df,
        target_users=TARGET_USERS,
        min_total_interactions=MIN_TOTAL_INTERACTIONS,
        test_frac=TEST_FRAC,
        val_frac=VAL_FRAC,
        seed=SPLIT_SEED,
    )
    N_USERS = meta["n_users"]
    N_ITEMS = meta["n_items"]

    # Save
    os.makedirs(SAMPLED_DATA_DIR, exist_ok=True)
    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path,     index=False)
    test_df.to_csv(test_path,   index=False)
    pd.DataFrame([
        {"key": "n_users",                "value": N_USERS},
        {"key": "n_items",                "value": N_ITEMS},
        {"key": "n_train",                "value": meta["n_train"]},
        {"key": "n_val",                  "value": meta["n_val"]},
        {"key": "n_test",                 "value": meta["n_test"]},
        {"key": "n_users_with_test",      "value": meta["n_users_with_test"]},
        {"key": "target_users",           "value": TARGET_USERS},
        {"key": "min_total_interactions", "value": MIN_TOTAL_INTERACTIONS},
        {"key": "test_frac",              "value": TEST_FRAC},
        {"key": "val_frac",               "value": VAL_FRAC},
        {"key": "split_seed",             "value": SPLIT_SEED},
        {"key": "strategy",               "value": "top_k_random_per_user_split"},
    ]).to_csv(meta_path, index=False)

    density = meta["n_train"] / (N_USERS * N_ITEMS) * 100
    print(f"  Users              : {N_USERS}")
    print(f"  Items              : {N_ITEMS}")
    print(f"  Train              : {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print(f"  Users with test    : {meta['n_users_with_test']}")
    print(f"  Density            : {density:.4f}%")

Cache not found — running sampling + split pipeline...
  Users passing floor (>= 20 unique items): 1764 / 1765
  Users              : 1000
  Items              : 10354
  Train              : 63081 | Val: 15010 | Test: 25055
  Users with test    : 1000
  Density            : 0.6092%


## 3. Dataset & DataLoader

## 4. MF Model